# Banked $3000 Every Game? The Last-Callable Trap

On 2026-08-31 I fired a credited copy of a public Kaggriculture agent that had just won **93 of 96** games against my best active agent on a fixed pool of real ladder matchups. Twelve hours later it sat at rating **161**.

Its live episodes tell the whole story. Every one of the 11 games shows reward **3000.0** — the starting bank — while its opponents banked 30,661, 35,561, 92,122. The replays (`kaggleusercontent.com/episodes/<id>.json`) show status `ACTIVE` all game, 1.5 s of overage consumed at step 1 and never again, and this action on all 720 steps:

```
{'farmer': ['PASS'], 'hands': [], 'market': []}
```

The agent never played a turn. Nothing crashed. Locally it was perfect.

## The three lines that decide what Kaggle calls

`kaggle_environments/agent.py` resolves a submitted file like this:

```python
env = {}
exec(code_object, env)
return [v for v in env.values() if callable(v)][-1]
```

That is **the last callable in the module namespace** — and a Python dict keeps names in **first-binding** order. Re-defining `agent` at the bottom of your file does **not** move it to the end. If `agent` was first bound in an early section (a bundled base version, a fallback stub, an earlier iteration) and any helper function is defined *after* that point, the helper is what Kaggle calls. It gets `(obs, configuration)`, raises `TypeError`, and this engine's malformed-action path turns the raise into a silent PASS. Bank stays at $3000. Status stays `DONE`.

The agent that fooled me does exactly that: `agent` and `_kaggle_submission_entrypoint` are first bound in a `v51` section a thousand lines up, then re-bound at the end; the last *new* callable is a one-argument helper. My bench loaded it with `getattr(module, "agent")` — by **name** — so the pool never saw it.

Below: a 10-line reproduction, the one-line fix, and a checker you can run on your own `main.py` before you spend a submission.


In [ ]:
import os, json, time, inspect, textwrap
from kaggle_environments import make
from kaggle_environments.agent import get_last_callable

LIVE = {"actTimeout": 1, "episodeSteps": 720, "runTimeout": 1200}   # the live runner's configuration, read off a replay

# A file that binds `agent` EARLY (a stub), defines a helper LATER, then re-defines `agent` at the end.
BAD = textwrap.dedent("""
    from kaggle_environments.envs.kaggriculture.kaggriculture import agents as _builtin
    _starter = _builtin["starter"]

    def agent(obs, configuration=None):          # v1 stub, bound FIRST
        return {"farmer": ["PASS"], "hands": [], "market": []}

    def _first_shop(obs):                        # a helper, bound AFTER `agent`
        return ((obs or {}).get("town", {}) or {}).get("unlocked_shops", [])

    def agent(obs, configuration=None):          # the real agent -- RE-bound, position unchanged
        return _starter(obs)              # the built-in starter takes (obs) only
""")
open("bad_agent.py", "w").write(BAD)

fn = get_last_callable(open("bad_agent.py").read(), path="bad_agent.py")
print("Kaggle will call:", fn.__name__, inspect.signature(fn))

def play(paths, seed=1):
    env = make("kaggriculture", configuration={**LIVE, "seed": seed})
    env.run(paths)
    fin = env.steps[-1]
    banks = [float(s.reward or 0) for s in fin]
    passes = [sum(1 for st in env.steps[1:] if (st[i].action or {}).get("farmer") == ["PASS"]
                  and not (st[i].action or {}).get("market") and not (st[i].action or {}).get("hands")) for i in range(2)]
    return banks, [s.status for s in fin], passes

banks, statuses, passes = play(["bad_agent.py", "random"])
print(f"bad_agent.py vs random -> bank {banks[0]:.0f} (opponent {banks[1]:.0f}), status {statuses[0]}, PASS steps {passes[0]}/720")


**`Kaggle will call: _first_shop (obs)`** — and the bank is exactly $3000 after 720 turns of PASS, with a clean `DONE`. That is the live failure, reproduced in four seconds.

## The fix: bind a NEW name, last

Renaming or re-defining `agent` cannot fix it. Append a function with a name the file has never used, and let it delegate:


In [ ]:
GOOD = BAD + textwrap.dedent("""

    def kaggle_entrypoint(obs, configuration=None):   # NEW name, bound LAST -> this is what Kaggle calls
        return agent(obs, configuration)
""")
open("good_agent.py", "w").write(GOOD)

fn = get_last_callable(open("good_agent.py").read(), path="good_agent.py")
print("Kaggle will call:", fn.__name__, inspect.signature(fn))
banks, statuses, passes = play(["good_agent.py", "random"])
print(f"good_agent.py vs random -> bank {banks[0]:.0f} (opponent {banks[1]:.0f}), status {statuses[0]}, PASS steps {passes[0]}/720")


## Check your own `main.py` before you submit

Copy & Edit this notebook, attach your submission file as a Dataset (or paste its source into `MAIN_SOURCE`), and run the cell. It resolves your file **the way the runner does**, then plays one live-configuration episode in each seat against the built-in `random` agent. An agent that Kaggle actually calls moves its bank off $3000 and does not PASS every turn; an agent caught by the trap shows exactly $3000 and ~719 PASS steps in both seats. (The built-in `starter` is weak — it only banks ~3,500 — but it *acts*, which is the only thing this checks.)


In [ ]:
MAIN_PATH = "/kaggle/input/your-dataset/main.py"     # <- point this at your submission file, or
MAIN_SOURCE = ""                                      # <- paste its full source here instead

def check(path):
    fn = get_last_callable(open(path).read(), path=path)
    name = getattr(fn, "__name__", repr(fn))
    try:
        sig = inspect.signature(fn); nparams = len(sig.parameters)
    except Exception:
        sig, nparams = "<?>", 0
    print(f"loader resolves: {name}{sig}")
    ok = nparams >= 1
    for seat in (0, 1):
        paths = [path, "random"] if seat == 0 else ["random", path]
        banks, statuses, passes = play(paths, seed=7 + seat)
        # The trap's signature is exact: the bank never moves off $3000 and (almost) every turn is PASS.
        never_acted = banks[seat] == 3000.0 and passes[seat] >= 700
        good = statuses[seat] == "DONE" and not never_acted
        ok &= good
        print(f"seat {seat}: bank {banks[seat]:.0f} vs {banks[1-seat]:.0f} | status {statuses[seat]} | PASS steps {passes[seat]}/720 -> {'acted' if good else 'NEVER ACTED'}")
    print("VERDICT:", "PASS -- Kaggle calls a real policy" if ok else "FAIL -- Kaggle will NOT call the function you think it does")
    return ok

if MAIN_SOURCE.strip():
    open("main_under_test.py", "w").write(MAIN_SOURCE); check("main_under_test.py")
elif os.path.exists(MAIN_PATH):
    check(MAIN_PATH)
else:
    print("No file attached -- demonstrating on the two toy agents above.\n")
    print("--- bad_agent.py ---");  check("bad_agent.py")
    print("\n--- good_agent.py ---"); check("good_agent.py")


## What this does and does not test

- It tests **resolution** (which function Kaggle calls) and **basic play** (does the resolved function act, in both seats, under the live `actTimeout`/`runTimeout`). Those two things sank a 97%-pool agent to rating 161.
- It does **not** test speed on Kaggle's runner, which is slower than your laptop. An agent that spends more than 1 s per step there burns its 60 s overage and is stopped; that is a different failure with a different signature (`TIMEOUT`, not `DONE`).
- `random` is a weak opponent. Beating it proves the agent *acts*; it proves nothing about strength. Rank strength on a fixed pool of real ladder matchups, not here.

The lesson generalises past this competition: **any simulation submission is resolved by position, not by name.** If you copy, bundle, or iterate versions inside one file, the safest habit is the one above — a fresh, never-before-used name, bound last, that delegates to whatever you mean.
